In [0]:
# Importando funções

from pyspark.sql import functions as F

In [0]:
from pyspark.sql.functions import col, regexp_extract, regexp_replace, expr, trim

# 1. Lemos a tabela que veio da bifurcação da Bronze
df_vendas_bronze = spark.read.table("tabela_vendas_bronze")

# 2. Criamos o DataFrame só com o volume físico (Pizzas)
# Extraímos apenas o número e convertemos para Inteiro (int)
df_pizzas = (
    df_vendas_bronze
    .filter(col("descricao_valor").rlike("(?i)pizzas"))
    .withColumn("qtd_pizzas", regexp_extract(col("descricao_valor"), r"(\d+)", 1).cast("int"))
    .select("data", "qtd_pizzas")
)

# 3. Criamos o DataFrame só com o fluxo financeiro (Caixa)
# Filtramos tudo que NÃO tem a palavra "pizzas"
df_caixa = df_vendas_bronze.filter(~col("descricao_valor").rlike("(?i)pizzas"))

print("--- VOLUME DE PIZZAS ---")
display(df_pizzas)

print("--- FLUXO FINANCEIRO BRUTO ---")
display(df_caixa)

In [0]:
df_caixa_limpo = (
    df_caixa.withColumn("descricao_valor", F.translate(col("descricao_valor"), "áàâãéêíóôõúçÁÀÂÃÉÊÍÓÔÕÚÇ", "aaaaeeioooucAAAAEEIOOOUC"))
    # 1. OS BAND-AIDS: Arrumamos as bizarrices de digitação na força bruta
    .withColumn("descricao_valor_corrigida", regexp_replace(col("descricao_valor"), r"146,9,00", "146,90"))
    .withColumn("descricao_valor_corrigida", regexp_replace(col("descricao_valor_corrigida"), r",0,00", "0,00"))
    
    # 2. Extrai o método de pagamento (procura a palavra chave)
    .withColumn("metodo_pagamento", regexp_extract(col("descricao_valor_corrigida"), r"(?i)(dinheiro|cartao|pix|fiado|ifood|magao)", 1))
    
    # 3. EXTRAÇÃO CIRÚRGICA: Ignora palavras como "nego" ou "Babidi". Caça apenas o primeiro bloco que pareça Dinheiro (Ex: "55,00" ou "96")
    .withColumn("valor_sujo", regexp_extract(col("descricao_valor_corrigida"), r"(\d+(?:[\,\.]\d+)?)", 1))
    
    # 4. Limpeza (Converte vírgula para ponto)
    .withColumn("valor_limpo", regexp_replace(col("valor_sujo"), ",", "."))
    
    # 5. O Cast Final (Transforma em número matemático)
    .withColumn("valor_receita", expr("try_cast(valor_limpo AS double)"))
    
    # Seleciona as colunas prontas
    .select("data", "metodo_pagamento", "valor_receita", "descricao_valor") 
)

# Rodamos o detector de nulos novamente para ver se zeramos a lista!
df_nulos = df_caixa_limpo.filter(col("valor_receita").isNull())
display(df_nulos)

In [0]:
# Filtro blindado: Se não tem método de pagamento (como a linha do "Total"), joga fora!
df_caixa_limpo = df_caixa_limpo.filter(col("metodo_pagamento") != "")

#Removemos a coluna original suja do caixa, deixando só os dados limpos

df_caixa_final = df_caixa_limpo.drop("descricao_valor")

# 2. Salvamos as duas tabelas separadas e prontas para a Camada Gold
df_pizzas.write.mode("overwrite").format("delta").saveAsTable("tabela_silver_vendas_pizzas")
df_caixa_final.write.mode("overwrite").format("delta").saveAsTable("tabela_silver_caixa")

print("Tabelas da Camada Silver de Vendas salvas com sucesso!")